In [14]:
"""
Cell 1: Loading the Master Data and the AI Models
First, we need to bring in our unified master dataset from Phase 5, and then load our "Champion" Logistic Regression model and Scaler from Phase 4.

"""

# Cell 1
import pandas as pd
import numpy as np
import joblib
import warnings

warnings.filterwarnings('ignore')

# Define paths
MASTER_DATA_PATH = '../data/processed/master_customer_dataset.parquet'
MODEL_PATH = '../models/churn/lr_champion.pkl'
SCALER_PATH = '../models/churn/feature_scaler.pkl'

print("Loading Master Dataset and Predictive Models...")

# Load the data
df_master = pd.read_parquet(MASTER_DATA_PATH)

# Load the Champion Model and Scaler
try:
    lr_model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    print("AI Model and Scaler successfully loaded.")
except FileNotFoundError:
    raise FileNotFoundError("Could not find the model files. Ensure Phase 4 was completed and saved correctly.")

print(f"Master Dataset Shape: {df_master.shape}")
display(df_master.head())

Loading Master Dataset and Predictive Models...
AI Model and Scaler successfully loaded.
Master Dataset Shape: (3370, 10)


,CustomerID,Recency,Frequency,Monetary,Churn,Tenure,Velocity,AOV,ItemDiversity,predicted_90d_clv
0,12346,235,1,77183.60,1,235,235.000000,77183.600000,1,77.190201
1,12347,39,5,2790.86,0,277,55.400000,558.172000,82,655.103993
2,12348,158,3,1487.24,0,268,89.333333,495.746667,22,221.076831
3,12350,220,1,334.40,1,220,220.000000,334.400000,17,81.418184
4,12352,172,5,1561.81,0,206,41.200000,312.362000,26,531.907730


In [15]:
"""
Cell 2: Calculating Exact Churn Probabilities
In Phase 4, our model predicted a hard 1 (Churn) or 0 (Stay). For business strategy, that isn't granular enough. We need to know if a customer is an 85% risk or a 51% risk. We will use predict_proba to extract the exact percentage risk for every single user."""

# Cell 2
print("Calculating granular Churn Probabilities...")

# Isolate the exact features the model was trained on
# We drop ID, the actual Churn label, and the newly added predicted_90d_clv
features_for_prediction = df_master.drop(columns=['CustomerID', 'Churn', 'predicted_90d_clv'])

# Scale the features using our saved Phase 4 scaler
X_scaled = pd.DataFrame(scaler.transform(features_for_prediction), columns=features_for_prediction.columns)

# Get the probability of class 1 (Churn)
# predict_proba returns an array like [prob_class_0, prob_class_1]
df_master['Churn_Probability'] = lr_model.predict_proba(X_scaled)[:, 1]

# Let's look at the distribution of our risk scores
print("\n--- Churn Probability Distribution ---")
display(df_master['Churn_Probability'].describe())

Calculating granular Churn Probabilities...

--- Churn Probability Distribution ---


count    3.370000e+03
mean     4.291300e-01
std      2.132984e-01
min      6.382172e-19
25%      2.738377e-01
50%      4.697098e-01
75%      6.056548e-01
max      1.000000e+00
Name: Churn_Probability, dtype: float64

In [16]:
"""
Cell 3: Building the Retention Strategy Matrix
We are going to divide your customer base into four strategic quadrants. To do this, we need to set two thresholds:

1. Risk Threshold: We will flag anyone with a > 0.50 (50%) churn probability as "High Risk".

2. Value Threshold: We don't want to spend budget saving $5 customers. We will calculate the 75th percentile of your CLV and flag anyone in the top 25% as "High Value."

"""
# Cell 3
print("Building the Retention Strategy Matrix...")

# Define business thresholds
# High Value = Top 25% of predicted 90-day CLV
clv_threshold = df_master['predicted_90d_clv'].quantile(0.75)

# High Risk = Churn Probability > 50%
risk_threshold = 0.50

# Define the logical conditions for our 4 quadrants
conditions = [
    (df_master['Churn_Probability'] >= risk_threshold) & (df_master['predicted_90d_clv'] >= clv_threshold), 
    (df_master['Churn_Probability'] < risk_threshold) & (df_master['predicted_90d_clv'] >= clv_threshold),  
    (df_master['Churn_Probability'] >= risk_threshold) & (df_master['predicted_90d_clv'] < clv_threshold),  
    (df_master['Churn_Probability'] < risk_threshold) & (df_master['predicted_90d_clv'] < clv_threshold)    
]

# Name the business segments
choices = [
    'High-Risk Whales (Immediate Action)',
    'Loyal Champions (Reward/Upsell)',
    'At-Risk Regulars (Automated Win-back)',
    'Safe Regulars (Monitor)'
]

# Apply the conditions to create the new segment column
df_master['Segment'] = np.select(conditions, choices, default='Unknown')

# Count how many customers fall into each segment
segment_counts = df_master['Segment'].value_counts().reset_index()
segment_counts.columns = ['Customer Segment', 'Number of Customers']

print(f"\nValue Threshold (Top 25% CLV cutoff): ${clv_threshold:.2f}")
print("--- Customer Segmentation Distribution ---")
display(segment_counts)


Building the Retention Strategy Matrix...

Value Threshold (Top 25% CLV cutoff): $402.36
--- Customer Segmentation Distribution ---


,Customer Segment,Number of Customers
0,At-Risk Regulars (Automated Win-back),1505
1,Safe Regulars (Monitor),1022
2,Loyal Champions (Reward/Upsell),831
3,High-Risk Whales (Immediate Action),12


12 Customers. Out of an initial dataset of nearly 400,000 transactions and 3,370 total customers, your AI has perfectly isolated exactly 12 "High-Risk Whales".

- If you only used standard Churn Prediction (Phase 4): The marketing team would look at the data and see 1,517 customers about to leave (1,505 At-Risk Regulars + 12 High-Risk Whales). They would waste their budget sending generic 10% off coupons to all of them, losing money in the process.
- Because you added CLV (Phase 5): The business now knows to completely ignore those 1,505 low-value users. Instead, they can take that entire marketing budget and have an account manager personally call those 12 specific VIPs today. You just solved the exact problem you outlined in Chapter 1 of your report

We need to extract this "Hit List" so the marketing team can actually use it.

In [17]:
"""
Cell 4: Extracting the VIP Action List
Copy and run this code to pull those 12 specific customers and save them to a specialized CSV file.

"""
# Cell 4
import os

print("Extracting the High-Risk Whales Action List...")

# Filter for only our top priority segment
vip_hit_list = df_master[df_master['Segment'] == 'High-Risk Whales (Immediate Action)']

# Sort them by their exact predicted value so the team calls the most valuable one first
vip_hit_list = vip_hit_list.sort_values(by='predicted_90d_clv', ascending=False)

# Select only the actionable columns for the marketing team
actionable_columns = [
    'CustomerID', 
    'Churn_Probability', 
    'predicted_90d_clv', 
    'Recency', 
    'Frequency', 
    'Monetary'
]

final_export = vip_hit_list[actionable_columns]

# Ensure directory exists
os.makedirs('../data/processed/exports', exist_ok=True)

# Save to a highly accessible CSV format
EXPORT_PATH = '../data/processed/exports/VIP_Action_List.csv'
final_export.to_csv(EXPORT_PATH, index=False)

print(f"SUCCESS: VIP Action List saved to {EXPORT_PATH}")
display(final_export.head(12)) # Let's see all 12 of them!


Extracting the High-Risk Whales Action List...
SUCCESS: VIP Action List saved to ../data/processed/exports/VIP_Action_List.csv


,CustomerID,Churn_Probability,predicted_90d_clv,Recency,Frequency,Monetary
2392,16532,0.530635,1260.412943,45,2,4516.80
2840,17353,0.511944,1086.810453,39,2,1740.00
2479,16692,0.617913,735.046984,166,2,1276.00
779,13680,0.555238,734.925060,247,2,2391.59
844,13791,0.501621,650.644470,37,2,1516.00
3284,18142,0.515419,608.095543,67,2,1019.52
2483,16698,0.597250,554.459427,136,2,1998.00
962,14001,0.545105,439.740655,142,2,1309.14
1582,15073,0.515537,428.926616,95,2,861.96
642,13441,0.509810,406.469256,1,1,296.64


In [18]:
#Cell 5: The Interactive Retention Strategy Matrix
# Cell 5
import plotly.express as px

print("Generating the Clean & Spacious Strategy Matrix...")

# 1. Define a modern, minimalist color palette
# We make the VIPs pop, and let the rest fade elegantly into the background
color_map = {
    'High-Risk Whales (Immediate Action)': '#e63946',   # Vivid Red (Draws the eye)
    'Loyal Champions (Reward/Upsell)': '#8fbc8f',       # Muted, soft green
    'At-Risk Regulars (Automated Win-back)': '#f4a261', # Soft orange
    'Safe Regulars (Monitor)': '#e2e8f0'                # Light gray (keeps them visible but out of the way)
}

# 2. Create a clean, spacious scatter plot
fig = px.scatter(
    df_master,
    x='Churn_Probability',
    y='predicted_90d_clv',
    color='Segment',
    hover_name='CustomerID',
    hover_data={
        'Churn_Probability': ':.1%', 
        'predicted_90d_clv': ':$,.0f',
        'Segment': False # Hide to keep the hover box small and clean
    },
    title='<b>Customer Retention Strategy Matrix</b><br><sup>Clear View: Identifying the 12 VIPs</sup>',
    color_discrete_map=color_map,
    opacity=0.85
)

# 3. Standardize dot size for a clean look
fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color='white')))

# 4. Add delicate, minimalist threshold lines
fig.add_hline(
    y=clv_threshold, 
    line_dash="dash", 
    line_color="#94a3b8", 
    line_width=1.5,
    annotation_text="Top 25% Value Cutoff",
    annotation_position="bottom right",
    annotation_font=dict(color="#64748b", size=12)
)
fig.add_vline(
    x=risk_threshold, 
    line_dash="dash", 
    line_color="#94a3b8", 
    line_width=1.5,
    annotation_text="50% Risk Cutoff",
    annotation_position="top left",
    annotation_font=dict(color="#64748b", size=12)
)

# 5. Apply a spacious, breathable layout
fig.update_layout(
    template='plotly_white',
    xaxis=dict(
        title='<b>Churn Risk Probability</b>', 
        tickformat=".0%", 
        showgrid=False,       # Remove vertical grid lines for a cleaner look
        zeroline=False
    ), 
    yaxis=dict(
        title='<b>Predicted 90-Day CLV ($)</b>', 
        tickprefix="$", 
        showgrid=True, 
        gridcolor='#f8fafc',  # Make horizontal lines almost invisible
        zeroline=False
    ),   
    height=700,
    margin=dict(l=40, r=40, t=100, b=40), # Add physical breathing room around the edges
    font=dict(family="Helvetica, Arial, sans-serif", size=14, color="#334155"),
    
    # Move legend to the top right to free up vertical space
    legend=dict(
        title="",
        orientation="h",
        yanchor="bottom",
        y=1.02, 
        xanchor="right",
        x=1
    ),
    hoverlabel=dict(bgcolor="white", font_size=14, font_family="Helvetica")
)

# Render the clean plot
fig.show()

Generating the Clean & Spacious Strategy Matrix...


In [19]:
# Cell 6: Saving the final segmented dataset for the dashboard
import os

print("Saving the fully segmented dataset for the dashboard...")
MASTER_DATA_PATH = '../data/processed/master_customer_dataset.parquet'

# Overwrite the old master file with our new Churn_Probability and Segment columns
df_master.to_parquet(MASTER_DATA_PATH, index=False)

print("SUCCESS: Final data saved. The dashboard is ready.")
display(df_master.head(3)) # Just to confirm the columns are there

Saving the fully segmented dataset for the dashboard...
SUCCESS: Final data saved. The dashboard is ready.


,CustomerID,Recency,Frequency,Monetary,Churn,Tenure,Velocity,AOV,ItemDiversity,predicted_90d_clv,Churn_Probability,Segment
0,12346,235,1,77183.60,1,235,235.000000,77183.600000,1,77.190201,1.000000,At-Risk Regulars (Automated Win-back)
1,12347,39,5,2790.86,0,277,55.400000,558.172000,82,655.103993,0.170533,Loyal Champions (Reward/Upsell)
2,12348,158,3,1487.24,0,268,89.333333,495.746667,22,221.076831,0.488238,Safe Regulars (Monitor)
